# OCR Extraction & Engine Comparison

This notebook takes your preprocessed image and runs it through multiple OCR engines so you can compare accuracy yourself. Install each engine's library before running that section (noted in each cell) — you don't need all of them at once, start with Tesseract since it's already installed.

In [1]:
import sys
sys.path.insert(0, ".")  # adjust if image_preprocessing_v2.py is elsewhere
from image_preprocessing import preprocess_pipeline

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

import cv2

ModuleNotFoundError: No module named 'image_preprocessing'

## 1. Run preprocessing on your image

In [ ]:
input_path = "D:/stage Mobilis/OCR-registre-commerce/input/REGISTRE-FRONT.png"  # <-- change this
results = preprocess_pipeline(input_path, scale=2.0)
final_image = results["8_thinned"]

## 2. Tesseract — try multiple PSM modes, keep the best

**What changed vs. your original `ocr.py`:** instead of locking in `--psm 6` (assumes one uniform text block), we try several PSM modes and automatically keep the one Tesseract itself is most confident about. Your document is a form with a two-column layout, ruled boxes, and mixed text sizes — a single fixed PSM assumption is unlikely to be optimal for the whole page.

- `psm 3` — fully automatic page segmentation (no orientation detection): good general default for a full page with mixed layout
- `psm 4` — assumes a single column of text of variable sizes
- `psm 6` — assumes one uniform block of text (your original choice)
- `psm 11` — sparse text: find as much text as possible with no particular reading order — useful for forms with scattered labels/values

We use `pytesseract.image_to_data` (not just `image_to_string`) because it also returns a per-word confidence score, which lets us judge which PSM actually worked best instead of guessing.

In [ ]:
def ocr_with_confidence(image, psm, lang='ara+fra'):
    config = f'--oem 3 --psm {psm}'
    data = pytesseract.image_to_data(image, lang=lang, config=config, output_type=pytesseract.Output.DICT)
    confidences = [int(c) for c in data['conf'] if c != '-1']
    avg_conf = sum(confidences) / len(confidences) if confidences else 0
    text = pytesseract.image_to_string(image, lang=lang, config=config)
    return text, avg_conf


def extract_text_tesseract_best_psm(image, psm_list=(3, 4, 6, 11), lang='ara+fra'):
    best_text, best_conf, best_psm = "", -1, None
    for psm in psm_list:
        text, conf = ocr_with_confidence(image, psm, lang=lang)
        print(f"  psm {psm}: avg confidence = {conf:.1f}")
        if conf > best_conf:
            best_text, best_conf, best_psm = text, conf, psm
    print(f"-> Best: psm {best_psm} (confidence {best_conf:.1f})")
    return best_text

Run it:

In [ ]:
tesseract_text = extract_text_tesseract_best_psm(final_image)
print(tesseract_text)

## 3. EasyOCR

**Install:** `pip install easyocr`

**Important limitation to know about:** EasyOCR groups languages by shared character set. Arabic can only be combined with English (`['ar','en']`) — **not directly with French**. French needs its own separate pass (`['fr']`). This is a real EasyOCR restriction, not a bug below — for a mixed Arabic/French document we run it twice and merge the results by vertical position.

In [ ]:
import easyocr

def extract_text_easyocr(image):
    reader_ar = easyocr.Reader(['ar', 'en'], gpu=False)
    reader_fr = easyocr.Reader(['fr'], gpu=False)

    results_ar = reader_ar.readtext(image, detail=1)
    results_fr = reader_fr.readtext(image, detail=1)

    all_results = results_ar + results_fr
    all_results.sort(key=lambda r: r[0][0][1])  # sort top-to-bottom by y-coordinate

    lines = [f"{text}  (conf: {conf:.2f})" for (_, text, conf) in all_results]
    return "\n".join(lines)

Run it:

In [ ]:
easyocr_text = extract_text_easyocr(final_image)
print(easyocr_text)

## 4. PaddleOCR

**Install:** `pip install paddlepaddle paddleocr` (CPU build; see PaddleOCR docs for the GPU wheel if you have a CUDA GPU)

**Same limitation as EasyOCR:** PaddleOCR's standard PP-OCR recognition models are also grouped by script — `lang='ar'` and `lang='fr'` are separate models, so this also needs two passes for a mixed-script page.

Note: PaddleOCR's Python API has shifted across versions (older `.ocr()` vs. newer `.predict()`). If this errors on your installed version, check `pip show paddleocr` and the current usage docs — the snippet below matches the widely-used stable pattern as of early 2026.

In [ ]:
from paddleocr import PaddleOCR

def extract_text_paddleocr(image):
    ocr_ar = PaddleOCR(use_angle_cls=True, lang='ar')
    ocr_fr = PaddleOCR(use_angle_cls=True, lang='fr')

    result_ar = ocr_ar.ocr(image, cls=True)
    result_fr = ocr_fr.ocr(image, cls=True)

    lines = []
    for result in (result_ar, result_fr):
        if result and result[0]:
            for line in result[0]:
                text, conf = line[1]
                lines.append(f"{text}  (conf: {conf:.2f})")
    return "\n".join(lines)

Run it:

In [ ]:
paddleocr_text = extract_text_paddleocr(final_image)
print(paddleocr_text)

## 5. Side-by-side comparison

In [ ]:
print("=== TESSERACT ===")
print(tesseract_text)
print("\n=== EASYOCR ===")
print(easyocr_text)
print("\n=== PADDLEOCR ===")
print(paddleocr_text)

## Notes on combining engines (ensemble)

Once you have output from more than one engine, a simple and effective combination strategy for Step 4/5 later:
1. Run all engines on the same final preprocessed image.
2. For each expected *field* (once you build field extraction in Step 3), compare what each engine returned for that region.
3. Prefer the engine with the highest confidence score for that specific field, rather than picking one engine for the whole document — different engines tend to win on different field types (e.g. Tesseract often does fine on Latin dates/numbers, while PaddleOCR/EasyOCR tend to do better on Arabic script).
4. Log disagreements between engines — they're a good automatic signal for "this field needs manual review," which feeds directly into your validation step.